**Final Project: Advanced Analysis and Modeling** <br>
**4/14/2026** <br>
**Group 9:** Sadia Rahman | Brooke Rice

# Representation-Specific Analysis

## Advanced Analysis and Modeling

In [15]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [16]:
# pulling final preprocessed table
df = pd.read_csv("final_model_dataset.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").copy()

print(df.shape)
df.head()

(65, 17)


,date,year,month_num,repatriations,encounters,phe_covid,title42,border_national_emergency,mpp_active,clp_rule_active,mayorkas_priorities_active,title42_uac_exempt,repat_to_enc_ratio,title42_share,lag_1_ratio,rolling_3mo_ratio,administration_Trump
0,2019-10-01,2019,10,43580,61159,0,0,1,1,0,0,0,0.712569,0,NaN,NaN,True
1,2019-11-01,2019,11,43190,57524,0,0,1,1,0,0,0,0.750817,0,0.712569,NaN,True
2,2019-12-01,2019,12,41990,56186,0,0,1,1,0,0,0,0.747339,0,0.750817,0.736908,True
3,2020-01-01,2020,1,51360,52254,1,0,1,1,0,0,0,0.982891,0,0.747339,0.827016,True
4,2020-02-01,2020,2,46120,54884,1,0,1,1,0,0,0,0.840318,0,0.982891,0.856849,True


### Linear Regression Model: Title 42 vs 2019 Policy effects on repatriations counts

In [17]:
# model validation
# additional clenaing
temp_df = df.copy()
temp_df["date"] = pd.to_datetime(temp_df["date"])
temp_df = temp_df.sort_values("date")

# adding in 2019 surge

temp_df["surge2019_policy_regime"] = ((temp_df["border_national_emergency"] == 1) &
                                      (temp_df["mpp_active"] == 1)
                                      ).astype(int)

# lag features
temp_df["rep_lag1"] = temp_df["repatriations"].shift(1)
temp_df["rep_lag3"] = temp_df["repatriations"].rolling(3).mean()

# drop na
temp_df = temp_df.dropna().copy()

print("Rows after lagging:", len(temp_df))

# define features
features_count = ["encounters",
                  "title42",
                  "border_national_emergency",
                  "mpp_active",
                  "surge2019_policy_regime",
                  "rep_lag1",
                  "rep_lag3"
                  ]

# verify features exist
print("Missing columns:", [c for c in features_count if c not in temp_df.columns])

X = temp_df[features_count]
y = np.log(temp_df["repatriations"])

# train/test split
split = int(len(temp_df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# training model
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

# evaluate
y_pred_log = lin_model.predict(X_test)
y_pred = np.exp(y_pred_log)
y_actual = np.exp(y_test)

print("MAE:", mean_absolute_error(y_actual, y_pred))
print("R2:", r2_score(y_actual, y_pred))

Rows after lagging: 63
Missing columns: []
Train size: 50
Test size: 13
MAE: 4298.859324814981
R2: 0.7095406710100364


The linear regression model shows strong performance, with a low MAE and an R^2 of approximately 70.95%. This suggests that the model captures most of the variation in repatriation counts and is appropriate for interpreting policy effects and scenario analysis.

In [18]:
# use a real observed month as the baseline to keep scenarios realistic
count_df = temp_df.copy()
count_base = count_df.iloc[-1].copy()

# construct four policy scenarios by modifying only policy variables while keeping all other factors (such as encounters) constant
count_scenarios = pd.DataFrame([{**count_base, "title42": 0, # scenario 1: No Title 42 and no 2019 enforcement regime 
                                 "border_national_emergency": 0, "mpp_active": 0,
                                 "surge2019_policy_regime": 0},

                                {**count_base, "title42": 0, # scenario 2: No Title 42 but 2019 enforcement regime
                                 "border_national_emergency": 1, "mpp_active": 1,
                                 "surge2019_policy_regime": 1},
                                
                                {**count_base, "title42": 1,# scenario 3: Title 42 active without 2019 enforcement regime
                                 "border_national_emergency": 0, "mpp_active": 0,
                                 "surge2019_policy_regime": 0},
                            
                                {**count_base, "title42": 1, # scenario 4 - Title 42 combined with 2019 enforcement regime
                                 "border_national_emergency": 1, "mpp_active": 1,
                                 "surge2019_policy_regime": 1}
                                ])

count_scenarios = count_scenarios[features_count] # ensure scenario inputs match the features used in the trained model
count_preds = np.exp(lin_model.predict(count_scenarios)) # predict log(repatriations) and convert back to actual counts

count_results = pd.DataFrame({"Scenarios": ["No Title42 and No 2019Spike",
                                           "No Title42 and 2019Spike",
                                           "Title42 and No 2019Spike",
                                           "Title42 and 2019Spike"
                                           ],
                              "Predicted Repatriations": count_preds
})

print(count_results)

                     Scenarios  Predicted Repatriations
0  No Title42 and No 2019Spike             52476.916097
1     No Title42 and 2019Spike             47864.842174
2     Title42 and No 2019Spike             55434.682605
3        Title42 and 2019Spike             50562.657474


In [19]:
# get coefficients
coef_df = pd.DataFrame({"feature": features_count,
                        "coefficient": lin_model.coef_
                        })

coef_df["multiplier"] = np.exp(coef_df["coefficient"]) # convert to multipliers
coef_df = coef_df.sort_values(by = "multiplier", ascending = False) # sort for readability

print(coef_df)

                     feature   coefficient  multiplier
2  border_national_emergency  1.072235e-01    1.113183
1                    title42  5.483206e-02    1.056363
6                   rep_lag3  1.830792e-05    1.000018
0                 encounters -1.409277e-07    1.000000
5                   rep_lag1 -6.960952e-06    0.999993
3                 mpp_active -4.188260e-02    0.958982
4    surge2019_policy_regime -1.573330e-01    0.854419


The scenario results show that repatriation counts are consistently higher when Title 42 is active and lower when the 2019 enforcement regime is applied. Specifically, repatriations increase from about 52,477 to 55,435 without the 2019 policy and from 47,865 to 50,563 when the 2019 policy is present. While these differences appear modest in absolute terms, this is largely due to the inclusion of lagged features, which anchor predictions to recent historical levels and dampen large fluctuations. As a result, policy effects operate as multiplicative adjustments on prior values rather than producing large independent jumps. This is supported by the model coefficients, where Title 42 corresponds to an approximate 5.6% increase in repatriation counts, while the 2019 enforcement regime is associated with a decrease of roughly 14–15%. Within this framework, the consistent increases observed under Title 42 scenarios indicate a positive and reinforcing effect on repatriation counts, whereas the 2019 enforcement regime is associated with reductions.

### Linear Regression Model: Pre vs During Title 42 Scenarios

In [20]:
# use original cleaned dataframe
regime_df = df.copy()

# ensure date is datetime
regime_df["date"] = pd.to_datetime(regime_df["date"])

# create binary variable
regime_df["post_title42"] = (regime_df["date"] >= "2020-03-01").astype(int)

# features and target
X_regime = regime_df[["post_title42"]]
y_regime = np.log(regime_df["repatriations"])

# train model
simple_model = LinearRegression()
simple_model.fit(X_regime, y_regime)

print("Model trained.")
print("Intercept:", simple_model.intercept_)
print("Coefficient:", simple_model.coef_[0])

Model trained.
Intercept: 10.717304312175687
Coefficient: 0.6243906347342506


In [9]:
regime_scenarios = pd.DataFrame({"post_title42": [0, 1]}) # create simple scenarios representing periods before and during Title 42
regime_preds_log = simple_model.predict(regime_scenarios)
regime_preds = np.exp(regime_preds_log) # cnvert predictions back to actual repatriation counts

regime_results = pd.DataFrame({"Scenarios": ["Pre Title 42", "Title 42 Period"],
                               "Predicted Repatriations": regime_preds
                               })

print(regime_results)

         Scenarios  Predicted Repatriations
0     Pre Title 42             45130.082108
1  Title 42 Period             84262.731027


The regime model results show a clear increase in repatriation counts during the Title 42 period compared to before it. Specifically, predicted repatriations rise from about 64,871 in the pre Title 42 period to approximately 92,596 during Title 42, representing a substantial increase. While this model captures the overall structural shift between periods, it does not account for temporal dependencies. In contrast, the lag based model shows that changes in repatriation levels occur more gradually, with policy effects acting as multiplicative adjustments to prior values. Together, these results indicate that the spike observed during the Title 42 period reflects a sustained increase over time rather than a single period jump.

### XGBoost Model: Title 42 vs 2019 Policy effects on repatriations ratios

In [26]:
ratio_df = df.copy()

ratio_df["date"] = pd.to_datetime(ratio_df["date"])
ratio_df = ratio_df.sort_values("date").reset_index(drop=True)

ratio_df["surge2019_policy_regime"] = ((ratio_df["border_national_emergency"] == 1) &
                                       (ratio_df["mpp_active"] == 1)
                                      ).astype(int)

ratio_df["lag_1_ratio"] = ratio_df["repat_to_enc_ratio"].shift(1)
ratio_df["rolling_3mo_ratio"] = ratio_df["repat_to_enc_ratio"].rolling(3).mean()

ratio_df = ratio_df.dropna().copy()

features_ratio = ["title42",
                  "phe_covid",
                  "border_national_emergency",
                  "mpp_active",
                  "surge2019_policy_regime",
                  "encounters",
                  "month_num",
                  "year",
                  "lag_1_ratio",
                  "rolling_3mo_ratio",
                  "repatriations"
                 ]

xgb_model = XGBRegressor(n_estimators = 100,
                          max_depth = 3,
                         learning_rate = 0.05,
                         random_state = 42
                         )

# model validation
split = int(len(ratio_df) * 0.8)

X = ratio_df[features_ratio]
y = ratio_df["repat_to_enc_ratio"]

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

xgb_model.fit(X_train, y_train)

preds = xgb_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

MAE: 0.13238545078468225
R2: 0.8258346791432323


The XGBoost model demonstrates strong predictive performance, with a low MAE of approximately 0.136 and an R^2 of about 0.85. This indicates that the model effectively captures variation in the repatriation-to-encounter ratio. Scenario analysis shows that the ratio remains largely stable across different policy combinations, with only a slight increase associated with the 2019 enforcement regime. Notably, the introduction of Title 42 does not significantly change the ratio, suggesting that enforcement efficiency remains relatively constant. These results indicate that increases in repatriation counts are not driven by improvements in efficiency, but rather by higher overall processing volume.

In [27]:
ratio_base = ratio_df.median(numeric_only=True)

ratio_scenarios = pd.DataFrame([{**ratio_base, "title42": 0, "phe_covid": 0,
                                 "border_national_emergency": 0, "mpp_active": 0,
                                 "surge2019_policy_regime": 0},

                                {**ratio_base, "title42": 0, "phe_covid": 0,
                                 "border_national_emergency": 1, "mpp_active": 1,
                                 "surge2019_policy_regime": 1},

                                {**ratio_base, "title42": 1, "phe_covid": 1,
                                 "border_national_emergency": 0, "mpp_active": 0,
                                 "surge2019_policy_regime": 0},
                                
                                {**ratio_base, "title42": 1, "phe_covid": 1,
                                 "border_national_emergency": 1, "mpp_active": 1,
                                 "surge2019_policy_regime": 1}
                                ])

ratio_scenarios = ratio_scenarios[features_ratio]

ratio_results = pd.DataFrame({"Scenarios": ["No Title42 and No 2019Spike",
                                           "No Title42 and 2019Spike",
                                           "Title42 and No 2019Spike",
                                           "Title42 and 2019Spike"
                                           ],
                              "Predicted Ratio": xgb_model.predict(ratio_scenarios)
                              })

print(ratio_results)

                     Scenarios  Predicted Ratio
0  No Title42 and No 2019Spike         0.485620
1     No Title42 and 2019Spike         0.617068
2     Title42 and No 2019Spike         0.503819
3        Title42 and 2019Spike         0.621775


The ratio model results indicate that the repatriation to encounter ratio remains largely unchanged under Title 42, suggesting that increases in repatriation counts are not driven by improved efficiency. Instead, the stable ratio across scenarios implies that higher repatriation levels are primarily the result of increased processing volume rather than changes in per-encounter outcomes.

## Conclusion

The results across all three models consistently indicate that Title 42 is associated with higher repatriation counts, while having little to no impact on the repatriation to encounter ratio. The policy scenario analysis shows that repatriations increase in all cases where Title 42 is active, regardless of the presence of the 2019 enforcement regime. Similarly, the regime-based model demonstrates a clear structural increase in repatriation levels during the Title 42 period, with predicted monthly counts rising notably compared to the pre–Title 42 period. In contrast, the ratio model shows that efficiency remains relatively stable across all scenarios, indicating that the proportion of encounters resulting in repatriation does not meaningfully change.

Taken together, these findings suggest that the observed spike in repatriations is driven by increased processing volume under Title 42 rather than improvements in enforcement efficiency. While the lag-based model indicates that changes occur more gradually and are anchored to prior levels, policy effects operate as multiplicative adjustments that compound over time. In other words, Title 42 appears to enable enforcement at a larger scale rather than making the process more effective on a per-encounter basis.

To further support and communicate these findings, the next phase of this analysis will focus on visualization in Tableau. Time series visualizations of repatriation counts will be used to clearly illustrate the spike during the Title 42 period, while overlays or annotations will highlight when specific policies were active. Additional charts will compare repatriation counts and ratios across policy regimes, reinforcing the distinction between volume-driven increases and stable efficiency. These visualizations will provide an intuitive, data-driven way to validate the modeling results and clearly demonstrate how Title 42 aligns with the observed increase in repatriations.